# 06 · Scoring + MLflow + Contrato 2
**Corre en Databricks.** Depende de los notebooks 01–04 (rama `feat/modelo`).

Objetivo:
1. Reentrenar el modelo final (LightGBM, params de Optuna) sobre el train completo.
2. Registrarlo en MLflow (params + métricas + artefacto + signature).
3. Scorear **todas** las sesiones limpias de la Gold → probabilidad calibrada.
4. Asignar el segmento de clustering a cada sesión.
5. Persistir el **Contrato 2** (`user_session`, `prob_calibrada`, `segmento`, `segmento_nombre`) en Delta → lo consumen Kelly (tablero) y Yeison (A/B).


In [ ]:
# ── §0  CONFIGURACIÓN ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import average_precision_score, brier_score_loss
import lightgbm as lgb
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

# ── paths ──────────────────────────────────────────────────────────────────
GOLD_PATH    = "/Volumes/workspace/default/e_commerce/gold_snapshot"
TABLA_SALIDA = "workspace.default.contrato2"         # tabla Delta de salida

# ── constantes ─────────────────────────────────────────────────────────────
RANDOM_STATE = 42
TRAIN_CORTE  = "2019-11-23"     # split Opción C
K_FINAL      = 5                # clusters del notebook 04
SAMPLE_KM    = 300_000          # muestra para re-fit KMeans

# ── best params de Optuna (notebook 02, Gold 14-17) ────────────────────────
BEST_PARAMS = dict(
    n_estimators      = 684,
    learning_rate     = 0.0197,
    num_leaves        = 211,
    max_depth         = 9,
    min_child_samples = 197,
    subsample         = 0.9247,
    colsample_bytree  = 0.8086,
    reg_lambda        = 0.8359,
    n_jobs            = -1,
    random_state      = RANDOM_STATE,
    verbose           = -1,
)

# ── features ───────────────────────────────────────────────────────────────
CLUSTER_FEATURES = ["total_views","distinct_products_viewed","brands_compared",
                    "categories_explored","categories_explored_cid",
                    "browsing_duration_sec","avg_price_viewed","max_price_viewed",
                    "electronics_view_share","revisit_intensity",
                    "views_per_minute","avg_inter_event_sec"]

MODEL_FEATURES = CLUSTER_FEATURES + ["day_of_week","is_weekend",
                                      "sin_navegacion_previa","hour_sin","hour_cos"]
print("Config OK")
print(f"Model features: {len(MODEL_FEATURES)} | Cluster features: {len(CLUSTER_FEATURES)}")


## §1  MLflow — configuración
⚠️ **Crítico en Databricks Free Edition:** `set_tracking_uri` y `set_registry_uri`
deben ir **antes** de `set_experiment`, o MLflow no encuentra el servidor gestionado.


In [ ]:
# ── §1  MLFLOW SETUP ───────────────────────────────────────────────────────
mlflow.set_tracking_uri("databricks")
try:
    mlflow.set_registry_uri("databricks-uc")   # Unity Catalog (si disponible)
except Exception:
    pass                                        # Community Edition sin UC: ignorar

EXPERIMENT_NAME = "/Users/tu_email@databricks.com/propension_g8"  # ← pon tu email
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Experimento: {EXPERIMENT_NAME}")


## §2  Carga Gold + cuarentena
Lee el snapshot de la Gold (14–17 nov cuarentenados por bandera).
El filtro `label_window_corrupt == 0` excluye la ventana corrupta automáticamente.


In [ ]:
# ── §2  CARGA ──────────────────────────────────────────────────────────────
gold = pd.read_parquet(GOLD_PATH)
gold = gold[gold["label_window_corrupt"] == 0].copy()
print(f"Sesiones limpias: {len(gold):,} | tasa compra: {gold['target_purchase'].mean():.4f}")


## §3  Feature engineering + split
Misma transformación que el notebook 02: hora del día como features cíclicas
(`hour_sin` / `hour_cos`) para que las 23 h y las 0 h queden contiguas.


In [ ]:
# ── §3  FEATURES + SPLIT ───────────────────────────────────────────────────
gold["hour_sin"] = np.sin(2 * np.pi * gold["session_hour"] / 24)
gold["hour_cos"] = np.cos(2 * np.pi * gold["session_hour"] / 24)

mask_train = pd.to_datetime(gold["session_date"]) <= pd.Timestamp(TRAIN_CORTE)
X       = gold[MODEL_FEATURES].astype("float64")
y       = gold["target_purchase"]
X_train, y_train = X[mask_train],  y[mask_train]
X_test,  y_test  = X[~mask_train], y[~mask_train]

spw = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"scale_pos_weight: {spw:.2f}")


## §4  Fit modelo final
Entrena LightGBM con los mejores params de Optuna sobre el **train completo**
y calibra con `CalibratedClassifierCV` (cv=3, isotonic), igual que en el notebook 02.
*Nota: este paso tarda varios minutos en la Gold completa.*


In [ ]:
# ── §4  FIT MODELO FINAL ───────────────────────────────────────────────────
gbm_final = lgb.LGBMClassifier(**BEST_PARAMS, scale_pos_weight=spw)
modelo = CalibratedClassifierCV(gbm_final, method="isotonic", cv=3)
modelo.fit(X_train, y_train)
print("Modelo calibrado entrenado ✓")


## §5  Registro en MLflow
Guarda: parámetros, métricas de test, el artefacto del modelo y la signature
(esquema entrada/salida). La signature permite que otros notebooks/servicios carguen
el modelo y validen sus inputs automáticamente.


In [ ]:
# ── §5  MLFLOW LOG ─────────────────────────────────────────────────────────
prob_test = modelo.predict_proba(X_test)[:, 1]
pr_auc    = average_precision_score(y_test, prob_test)
brier     = brier_score_loss(y_test, prob_test)
print(f"Test PR-AUC: {pr_auc:.4f} | Brier: {brier:.4f}")

with mlflow.start_run(run_name="lgbm_propension_final_g8"):
    # params
    mlflow.log_params(BEST_PARAMS)
    mlflow.log_param("scale_pos_weight", round(float(spw), 2))
    mlflow.log_param("calibracion",      "isotonic_cv3")
    mlflow.log_param("split_corte",      TRAIN_CORTE)
    mlflow.log_param("cuarentena",       "14-17 nov (label_window_corrupt)")
    # métricas
    mlflow.log_metric("cv_pr_auc_optuna", 0.1235)   # best value de Optuna
    mlflow.log_metric("test_pr_auc",     round(pr_auc, 4))
    mlflow.log_metric("test_brier",      round(brier,  4))
    # modelo + signature
    sig = infer_signature(X_test, prob_test)
    mlflow.sklearn.log_model(
        modelo, "propension_model",
        signature        = sig,
        input_example    = X_test.head(5),
        registered_model_name = "propension_compra_g8"   # ← nombre en el Registry
    )
    run_id = mlflow.active_run().info.run_id
print(f"Run registrado: {run_id}")


## §6  Scoring batch — todas las sesiones limpias
Asigna una probabilidad calibrada a **cada una** de las sesiones de la Gold
(no solo al test: en producción se scorea todo para que el tablero y el A/B
puedan consumir la probabilidad de cualquier sesión).


In [ ]:
# ── §6  SCORING BATCH ──────────────────────────────────────────────────────
# Nota: ~19.7 M filas × 3 modelos calibrados — puede tardar 3-8 min con n_jobs=-1
prob_full = modelo.predict_proba(X)[:, 1]
print(f"Sesiones scoreadas: {len(prob_full):,}")
print(f"Prob media: {prob_full.mean():.4f} | p90: {np.percentile(prob_full, 90):.4f}")


## §7  Asignar segmento (KMeans k=5)
Re-ajusta el KMeans con los mismos hiperparámetros y semilla que el notebook 04
(determinista con `n_init=10, random_state=42`) y asigna el segmento a cada sesión.
Luego mapea los números de cluster a nombres de negocio usando el perfil de cada uno.


In [ ]:
# ── §7  SEGMENTACIÓN ───────────────────────────────────────────────────────
# Re-fit escalador + KMeans (determinista: misma semilla que notebook 04)
idx_km = np.random.RandomState(RANDOM_STATE).choice(len(X), SAMPLE_KM, replace=False)
Xclus  = X[CLUSTER_FEATURES].astype("float64")
scaler = StandardScaler().fit(Xclus.iloc[idx_km])
Xs     = scaler.transform(Xclus)
km     = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10).fit(Xs[idx_km])
segmento_num = km.predict(Xs)
print("Clusters asignados:", dict(zip(*np.unique(segmento_num, return_counts=True))))

# Mapeo número → nombre de negocio usando medias de features clave
perfil_06 = (pd.DataFrame({"seg": segmento_num})
             .assign(electronics=X["electronics_view_share"].values,
                     precio=X["avg_price_viewed"].values,
                     vistas=X["total_views"].values)
             .groupby("seg").agg(electronics=("electronics",  "mean"),
                                  precio     =("precio",       "mean"),
                                  vistas     =("vistas",       "mean"))
             .round(2))

# Regla de mapeo alineada con el perfilado del notebook 04
def nombre_seg(row):
    if row["electronics"] > 0.90:  return "electronica_gama_media"
    if row["electronics"] > 0.60:  return "electronica_premium"
    if row["vistas"]      > 10:    return "explorador"
    if row["electronics"] < 0.05:  return "general_bajo_valor"
    return "anonimo"

perfil_06["nombre"] = perfil_06.apply(nombre_seg, axis=1)
print(perfil_06)
mapa_nombre = perfil_06["nombre"].to_dict()
segmento_nombre = np.array([mapa_nombre[s] for s in segmento_num])


## §8  Contrato 2 → Delta
Construye el dataframe del Contrato 2 y lo persiste como tabla Delta en Unity Catalog.
**Kelly** (tablero Power BI) y **Yeison** (A/B test) lo consumen con
`spark.table("workspace.default.contrato2")`.


In [ ]:
# ── §8  CONTRATO 2 ─────────────────────────────────────────────────────────
contrato2 = pd.DataFrame({
    "user_session"    : gold["user_session"].values,
    "prob_calibrada"  : prob_full.astype("float32"),
    "segmento"        : segmento_num.astype("int32"),
    "segmento_nombre" : segmento_nombre,
})
print(contrato2.head())
print(f"\nTotal: {len(contrato2):,}")
print(contrato2["segmento_nombre"].value_counts())

# Persistir en Delta (via Spark)
spark_df = spark.createDataFrame(contrato2)
(spark_df.write
 .format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(TABLA_SALIDA))
print(f"\n✅ Contrato 2 persistido en: {TABLA_SALIDA}")


## §9  Verificación final
Comprueba que la tabla quedó bien: cuenta de filas, distribución de segmentos,
rango de probabilidades y tasa base implícita del modelo.


In [ ]:
# ── §9  VERIFICACIÓN ───────────────────────────────────────────────────────
df_ver = spark.table(TABLA_SALIDA).toPandas()
print(f"Filas en Delta: {len(df_ver):,}")
print(f"\nDistribución de segmentos:")
print(df_ver["segmento_nombre"].value_counts(normalize=True).mul(100).round(1).to_string())
print(f"\nRango prob_calibrada: [{df_ver['prob_calibrada'].min():.4f}, {df_ver['prob_calibrada'].max():.4f}]")
print(f"Prob media (≈ tasa base limpia): {df_ver['prob_calibrada'].mean():.4f}")
print("\n✅ Contrato 2 verificado. Listo para Kelly (tablero) y Yeison (A/B).")
